# Chapter 38: a financial ML pipeline

Fresh examples check labels and mature training rows, compare validation schemes on noise, expose full-sample feature-selection leakage and fit a constructed Ridge panel. Shared logic is in ml_pipeline. Signal time is $D_k$, trade/label start is $D_{k+\ell}$, and label end/availability is $D_{k+\ell+h}$ in this particular constructed calendar. Training requires label availability no later than fit time; real settlement delays would need their own field.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch38_ml_pipeline_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import ml_pipeline as mp
ch38 = load_chapter('ch38_ml_pipeline')

## 1. Three assets, 12 days, lag 1 and horizon 2

At fit date $D_7$, signal indices through 4 are eligible. Excluding indices 5 and 6 removes six panel rows, not six dates.

In [ ]:
hand = ch38.hand_example()
lab = pd.DataFrame(hand["labels"]).pivot(index="k", columns="asset", values="y")
display(lab)
print("fit k", hand["fit_time_k"], "used", hand["used_train_dates_k"], "excluded rows", hand["n_excluded_unavailable"])
assert hand["used_train_dates_k"] == [1, 2, 3, 4] and hand["n_excluded_unavailable"] == 6

## 2. Noise features under different validation schemes

Compare shuffled K-fold, date-block K-fold and date walk-forward. Shuffling is an intentionally wrong diagnostic, never a model-selection rule here. Multiple fits may take tens of seconds or more depending on the machine; this revision has not timed them.

In [ ]:
sch = ch38.validation_schemes(seed=3801)
df = pd.DataFrame(sch["results"]).T.drop(columns="n_walk_forward_folds") * 100
display(df.round(2))
assert df.loc["knn", "shuffled_kfold_r2_vs_zero"] > 40 and df.loc["knn", "walk_forward_r2_vs_zero"] < 0

## 3. Feature-selection leakage

Select ten of 200 pure-noise features either on the full sample or inside the training pipeline. The full-sample result uses test information and is a negative control, not deployable performance.

In [ ]:
leak = ch38.selection_leak(seed=3802)
display(pd.DataFrame(leak["results"]).T.round(5))
assert leak["results"]["full_sample_selection"]["r2_vs_zero"] > leak["results"]["train_only_selection"]["r2_vs_zero"]

## 4. Ridge date walk-forward and calibration

Distinguish pooled levels from demeaned cross-sections. Pooled R² may be negative while cross-sectional R² and IC are positive. Positive cross-sectional calibration is a condition for turning a directional ranking into the chapter's positively oriented position rule; IC alone does not certify magnitude calibration. Inspect fold exclusions and chosen alpha, then save one new prediction-decile versus demeaned five-day-return chart.

In [ ]:
pc = ch38.panel_case(seed=3803)
m = pc["metrics"]
print({k: round(m[k], 5) for k in ("oos_r2_vs_zero", "oos_r2_vs_train_mean", "ic_mean", "ic_t", "hit_rate")})
print("cross-sectional R2", round(pc["cross_sectional_r2_demeaned"], 5), "slopes", round(pc["calibration_slope_y_on_pred"], 3),
      round(pc["cross_sectional_calibration_slope"], 3), "alphas", pc["chosen_alpha_by_fold"])
assert pc["excluded_rows_per_fold"] == [200]
dec = np.array(pc["within_date_decile_mean_demeaned_y"]) * 100
plt.bar(range(1, 11), dec); plt.axhline(0, color="grey", lw=0.6)
plt.xlabel("within-date prediction decile"); plt.ylabel("mean demeaned 5-day return (%)"); plt.title("CONSTRUCTED"); plt.savefig(RUN_DIR / "ch38_ml_pipeline_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()